# E1 · Feature importances with the wrong column names

**Type**: Find the bug · **Level**: ●●○ independent · **Time**: about 40 minutes · **core path ★**

## Objective

Find out why a report says the largest importance belongs to the column `pclass` when the model mostly uses **sex**: the colleague assigned the importances of the pipeline to the names of the original columns, but the `ColumnTransformer` has **reordered the columns** (categoricals first, then numerics).
Fix it with `get_feature_names_out()`.

## Dataset

`ds.titanic()`: 9 columns (`pclass, sex, age, siblings_spouses, parents_children, fare, port, deck, title`), stratified 75/25 split. Pipeline: `ColumnTransformer` that puts the categoricals **first** (`sex, port, deck, title`, with `OrdinalEncoder`) and the numerics **after** (`pclass, age, ...`), followed by `DecisionTreeClassifier(max_depth=4, min_samples_leaf=5)`.

## Educational purpose

Recognize a frequent bug in reports: when the pipeline reorders, adds (one-hot) or removes columns, position `i` of the importances **no longer** corresponds to column `i` of `X`. If the number of columns stays the same, no error flags the problem (and the report is published wrong).

## Task

1. Run the colleague's cell: `top_bug` (name of the most important feature according to the report) and `imp_bug`.
2. Save `pipeline_names` (list `get_feature_names_out()` of the preprocessing part) and `different_positions` (how many positions `i` have `X_tr.columns[i] != pipeline_names[i]` without the prefix `cat__`/`num__`).
3. Fix it: `imp_ok` (`pd.Series` with the importances and `pipeline_names` as index, with the prefixes removed) and `top_ok`.
4. Check with the permutation (`permutation_importance` on the pipeline, `n_repeats=10`, `random_state=0`, `scoring="roc_auc"`) that `top_ok` really is the most important column: save `top_permutation` (name).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier

df = ds.titanic()
X, y = df.drop(columns="survived"), df["survived"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
CAT = ["sex", "port", "deck", "title"]
NUM = ["pclass", "age", "siblings_spouses", "parents_children", "fare"]
prep = ColumnTransformer([("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OrdinalEncoder()), CAT),
                          ("num", SimpleImputer(strategy="median"), NUM)])
pipe = make_pipeline(prep, DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=0)).fit(X_tr, y_tr)

## Your work

### The colleague's code

In [ ]:
imp_bug = pd.Series(pipe[-1].feature_importances_, index=X_tr.columns)          # "the importances, with the column names"
top_bug = imp_bug.idxmax()
print(f"most important feature according to the report: {top_bug} ({imp_bug.max():.2f})")
imp_bug.round(3).sort_values(ascending=False).head(4)

### Your diagnosis and the fix

In [ ]:
pipeline_names = None
different_positions = None

In [ ]:
imp_ok = None
top_ok = None
top_permutation = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`pipeline_names = list(pipe[:-1].get_feature_names_out())`; without the prefix: `[n.split("__", 1)[1] for n in pipeline_names]`. Permutation: `pd.Series(permutation_importance(pipe, X_te, y_te, n_repeats=10, random_state=0, scoring="roc_auc").importances_mean, index=X_te.columns).idxmax()` (here the order is that of `X_te`, because the permutation acts on the input columns of the pipeline).'):
    check(top_bug == "pclass", "the colleague's report says the most important feature is `pclass`", "don't modify the colleague's code")
    check(pipeline_names is not None and len(pipeline_names) == 9 and pipeline_names[0].startswith("cat__sex") and different_positions >= 6, "the pipeline has 9 columns in a different order (cat before num): at least 6 different positions", "pipe[:-1].get_feature_names_out()")
    check(imp_ok is not None and top_ok == "sex" and imp_ok.max() > 0.4 and not any(str(n).startswith(("cat__", "num__")) for n in imp_ok.index), "the most important feature is `sex` (over 0.4); the names have no prefix", "remove 'cat__' and 'num__' from the names")
    check(close(imp_ok.sum(), 1.0) and set(imp_ok.index) == set(X_tr.columns), "same columns as X, importances that sum to 1", "reindex with the correct names")
    check(top_permutation == top_ok, "the permutation confirms: the most important column is `sex`", "permutation_importance(pipe, X_te, y_te, ...) has the same names as X_te.columns")
done()

---
The commented solution is in `solutions/E_find_the_bug/E1_importances_with_the_wrong_column_names.ipynb`. Open it after you pass the check, or after you have really tried.